# Study 813 — Maximum-Drawdown Anomaly 📉

**When a stock just took its deepest 12-month drawdown, does it keep sinking — or bounce?**

Sort a cross-section of stocks on each name's **trailing 12-month maximum drawdown** (the
largest peak-to-trough decline of its cumulative total return). The distress story says the
deepest-drawdown names keep **under-earning**; the reversal story says they **rebound**. We
take no prior — we sort into fractiles on a liquid US cross-section
(2010-01-04 → 2026-06-30, 50 names), measure the forward long-short spread, and
report the sign we actually find.

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — the deepest drawdowns of
all (names that fell and never recovered) are exactly what this survivor panel deletes, so
magnitudes are an upper bound.*


## 1. The idea in one picture

Each name's **maximum drawdown** is how far it fell from its own running peak over the last 12 months — a pure, price-based distress gauge. Rank the cross-section: the calm names (shallow drawdown) on one side, the recently battered names (deep drawdown) on the other. Then watch what happens *next*. Two rival stories: **distress** (the wounded keep bleeding) vs **reversal** (the wounded bounce). Only the tape settles it.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=-4.35, t_nw=-2.36, lo_bps=5.34, hi_bps=9.68, gross_sharpe=-0.59)
print('long calm / short distressed spread: %+.2f bps/day (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  calm book %+.2f bps vs distressed book %+.2f bps'
      % (R['lo_bps'], R['hi_bps']))
print('  spread = calm - distressed; NEGATIVE => the distressed names OUT-earned (a rebound)')

long calm / short distressed spread: -4.35 bps/day (NW t = -2.36)
  calm book +5.34 bps vs distressed book +9.68 bps
  spread = calm - distressed; NEGATIVE => the distressed names OUT-earned (a rebound)


## 2. Is the sort just lucky? A live synthetic control

We plant a *distress* effect in a seeded toy world (`edge>0`: fragile names have deep drawdowns AND low forward returns) and check the detector recovers it — and that it stays *silent* on the null (`edge=0`, drawdowns present but unpriced). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from max_drawdown import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=813, n_assets=40, n_days=1500))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.004, seed=813, n_assets=40, n_days=1500))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up POSITIVE = distress)' % planted['t_nw'])

null world   : spread NW t = -0.32  (should be ~0)
planted world: spread NW t = +9.03  (should light up POSITIVE = distress)


## 3. The honest verdict — a fragile *reversal*, not a distress premium

On this liquid mega-cap tape the long-calm / short-distressed spread is **-4.35 bps/day** with NW *t* = **-2.36** — the *negative* sign means the **distressed** (deep-drawdown) names actually **out-earned** the calm ones (distressed book **+9.68** vs calm **+5.34** bps). So on mega-caps it's the **reversal** story, not distress: the wounded bounced. But the effect is **not robust** — it lives in the 2,134-day 2018–2026 era (*t* = -2.09) and is absent 2010–2017 (*t* = -1.10). And it does not pay: the specified book loses money outright, and even the profitable *rebound* direction earns only +2.21 bps/day net at a fantasy 1 bp cost (*t* = +1.19, not significant) and dies at 5 bps. **Signal: Weak** (a significant but era-fragile rebound), **Tradability: Mirage**.